# Coupled Pendulums: Differential Equations and Physics-Informed Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nhungnh-dot/pendulumDE/blob/main/coupled_pendulum_lesson.ipynb)

> by Andrew Bennett and Nhung Nguyen

## Learning goals

In this notebook, we study a physical system made of **two pendulums connected by a spring**. Our main goal is to see how ideas from an elementary differential equations course can be combined with experimental data.

By the end of the notebook, you should be able to:

- identify the terms in a second-order differential equation;
- explain how damping and spring coupling affect the motion;
- understand why two connected pendulums lead to a **system of differential equations**;
- compare a model with measured data;
- understand the basic idea of a **Physics-Informed Neural Network (PINN)**; and
- see how unknown physical parameters can be estimated from data.

---

## 1. The differential-equation model

Let

- $\theta_1(t)$ = angle of the **left** pendulum,
- $\theta_2(t)$ = angle of the **right** pendulum.

The two pendulums have lengths $l_1$ and $l_2$. They are connected by a spring, so the motion of one pendulum affects the other.

For small angles, we use the approximation

$$
\sin(\theta)\approx \theta.
$$

With this approximation, the equations of motion are

$$
\theta_1''
+d_1\theta_1'
+\frac{g}{l_1}\theta_1
+\frac{K}{l_1}(l_1\theta_1-l_2\theta_2)
=0,
$$

$$
\theta_2''
+d_2\theta_2'
+\frac{g}{l_2}\theta_2
-\frac{K}{l_2}(l_1\theta_1-l_2\theta_2)
=0.
$$

Here a prime means differentiation with respect to time. For example,

$$
\theta_1'=\frac{d\theta_1}{dt},
\qquad
\theta_1''=\frac{d^2\theta_1}{dt^2}.
$$

### What do the terms mean?

Each equation contains familiar pieces:

- $\theta_i''$: **angular acceleration**;
- $d_i\theta_i'$: **damping**, which removes energy from the motion;
- $\dfrac{g}{l_i}\theta_i$: the restoring effect of **gravity**;
- the term containing $K$: the effect of the **spring connecting the two pendulums**.

The constant

$$
K=\frac{k}{m}
$$

combines the spring constant $k$ and the pendulum-bob mass $m$. We assume the two bobs have the same mass.

Because the equations contain both $\theta_1$ and $\theta_2$, they are **coupled**. We cannot solve for one pendulum without also considering the other.

### What will we do?

We have experimental measurements of the pendulums. We will use those measurements to estimate

$$
\theta_1(t),\qquad \theta_2(t),
$$

and the unknown physical parameters

$$
K,\qquad d_1,\qquad d_2.
$$

Instead of using only a traditional ODE solver, we will introduce a PINN. The important idea is simple:

> We want a predicted motion that both **matches the experimental data** and **approximately satisfies the differential equations**.

---

## ❓ Question 1: What Does Coupling Do?

Look again at the two differential equations:

$$
\theta_1''
+d_1\theta_1'
+\frac{g}{l_1}\theta_1
+\frac{K}{l_1}(l_1\theta_1-l_2\theta_2)=0,
$$

$$
\theta_2''
+d_2\theta_2'
+\frac{g}{l_2}\theta_2
-\frac{K}{l_2}(l_1\theta_1-l_2\theta_2)=0.
$$

### Your Task

Imagine that the spring is removed. This corresponds to setting

$$
K=0.
$$

Without doing any computation, simplify both differential equations.

### Questions to Answer

1. After setting $K=0$, does the equation for $\theta_1$ still contain $\theta_2$?
2. Does the equation for $\theta_2$ still contain $\theta_1$?
3. What does this tell you about the meaning of the word **coupled**?

**Your answer:**  
*Write here.*

---


In [ ]:
# ============================================================
# COUPLED PENDULUM PINN
# Experimental data + Fourier features + gradual physics
#
# Google Colab / PyTorch
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

## 2. Load the experimental data

The experiment was recorded as a table of measurements. This cell downloads the CSV file and gives names to its five columns:

- `t`: time in seconds;
- `a_right`, `b_right`: measured coordinates of the right pendulum;
- `a_left`, `b_left`: measured coordinates of the left pendulum.

In this notebook, we will use the **horizontal coordinates** `a_left` and `a_right`.

The remaining commands clean the table:

1. remove the original text/header rows,
2. convert the entries to numbers,
3. remove rows with missing or invalid values, and
4. sort the measurements by time.

The result is a numerical data set that we can use in the model.

In [ ]:
# ============================================================
# 2. LOAD EXPERIMENTAL DATA
# ============================================================

csv_url = (
    "https://raw.githubusercontent.com/"
    "ngochuytruong/data/main/data-diff-length.csv"
)
df = pd.read_csv(csv_url, header=None)

df.columns = [
    't',
    'a_right',
    'b_right',
    'a_left',
    'b_left'
]

# Remove text/header rows in original CSV
df = df.iloc[2:].reset_index(drop=True)

# Convert everything to numeric
df = df.apply(
    pd.to_numeric,
    errors='coerce'
)

# Remove invalid rows
df = df.dropna().reset_index(drop=True)

# Sort by time just in case
df = df.sort_values('t').reset_index(drop=True)

print(df.head())
print()
print("Number of experimental points:", len(df))

## 3. Extract the quantities we need

From the data table, we keep

- the time values `t`,
- the horizontal position `a_left` of the left pendulum,
- the horizontal position `a_right` of the right pendulum.

We also enter the known physical constants

$$
l_1=1.00\text{ m},\qquad
l_2=1.05\text{ m},\qquad
g=9.8\text{ m/s}^2.
$$

Throughout the notebook,

$$
\theta_1=\text{left pendulum},
\qquad
\theta_2=\text{right pendulum}.
$$

These constants will later appear directly in our differential equations.

In [ ]:
# ============================================================
# 3. EXTRACT POSITION DATA
# ============================================================

t = df['t'].to_numpy(dtype=np.float32)

a_right = df['a_right'].to_numpy(dtype=np.float32)
a_left  = df['a_left'].to_numpy(dtype=np.float32)

# Pendulum lengths [m]
l_left  = 1.00
l_right = 1.05

# Equation notation:
#
# theta1 = left
# theta2 = right
#
l1 = l_left
l2 = l_right

g = 9.8

## 4. Make the experiment start at $t=0$

The original recording does not necessarily begin at time zero. We replace every time value by

$$
t_{\text{new}}=t_{\text{old}}-t_{\text{first}}.
$$

After this change, the first observation occurs at

$$
t=0.
$$

This does **not** change the motion or the time between measurements. It only changes where we choose the clock to start.

This is similar to choosing a convenient initial time when solving an initial-value problem.

In [ ]:
# ============================================================
# 4. SHIFT TIME TO START AT ZERO
# ============================================================

t = t - t[0]

## 5. Convert horizontal position to angle

Our differential equations are written in terms of the angles $\theta_1$ and $\theta_2$, but the experiment gives horizontal position.

For a pendulum of length $L$,

$$
x=L\sin(\theta).
$$

For small angles,

$$
\sin(\theta)\approx\theta.
$$

Therefore,

$$
x\approx L\theta,
$$

so we can estimate the angle using

$$
\theta\approx\frac{x}{L}.
$$

For the two pendulums, the code uses

$$
\theta_1\approx\frac{a_{\text{left}}}{l_1},
\qquad
\theta_2\approx\frac{a_{\text{right}}}{l_2}.
$$

This is the same **small-angle approximation** that allows the pendulum equations to become linear.

The plot below lets us see the measured oscillations of both pendulums before we begin fitting a model.

In [ ]:
# ============================================================
# 6. CONVERT POSITION -> ANGLE
# ============================================================
#
# We assume a_left and a_right are horizontal x positions.
#
# First remove the equilibrium offset.
#
# For horizontal displacement:
#
#       x = L sin(theta)
#
# therefore
#
#       theta = arcsin(x/L)
#
# If your columns already represent displacement from x=0,
# subtracting the median will have very little effect.
#
# ============================================================

x_left = a_left
x_right = a_right

theta_left = x_left / l_left
theta_right = x_right / l_right

print("Theta left range:",
      theta_left.min(),
      theta_left.max())

print("Theta right range:",
      theta_right.min(),
      theta_right.max())

# Plot
plt.figure(figsize=(11, 5))

plt.plot(
    t,
    theta_left,
    label='Left experimental'
)

plt.plot(
    t,
    theta_right,
    '--',
    label='Right experimental'
)

plt.xlabel('Time [s]')
plt.ylabel(r'$\theta$ [rad]')
plt.title('Experimental Coupled Pendulum')
plt.legend()
plt.grid(True)

plt.show()

## 6. Separate the data into training and validation sets

We will not use every measured point to fit the model.

Instead, we divide the measurements by time:

- the **first 80%** are training data;
- the **last 20%** are validation data.

### Why do this?

The training data are the observations that the model is allowed to learn from.

The validation data are kept aside. After training, we ask:

> Can the model predict the later part of the experiment even though those measured angles were not used to fit it?

Because the last 20% occur later in time, this is a test of **prediction into the future**, or extrapolation.

The split is chronological rather than random:

$$
\boxed{\text{earlier times: training}}
\qquad
\boxed{\text{later times: validation}}.
$$

The known differential equations will still be enforced over the full time interval. However, the measured validation angles themselves are not used in the fitting loss.

The code also defines a typical angle size, called `theta_scale`, using only the training data. This scale helps keep numerical values at a reasonable size during training.

In [ ]:
# ============================================================
# 7. PREPARE PYTORCH DATA + CHRONOLOGICAL 80/20 SPLIT
# ============================================================

t_data = t.reshape(-1, 1)
theta1_data = theta_left.reshape(-1, 1)
theta2_data = theta_right.reshape(-1, 1)

t_min = float(t_data.min())
t_max = float(t_data.max())

# ------------------------------------------------------------
# First 80% of observations for training; last 20% for validation.
# df was sorted by t during preprocessing. NO random shuffling.
# ------------------------------------------------------------
train_fraction = 0.80
n_data = len(t_data)
assert n_data >= 2, "At least two observations are required to split the data."
assert np.all(np.diff(t_data[:, 0]) >= 0), "Time must be sorted."

n_train = int(np.floor(train_fraction * n_data))
train_indices = np.arange(n_train)
val_indices = np.arange(n_train, n_data)

# NumPy arrays used for plots and preprocessing.
t_train = t_data[train_indices]
t_val = t_data[val_indices]
theta1_train = theta1_data[train_indices]
theta2_train = theta2_data[train_indices]
theta1_val = theta1_data[val_indices]
theta2_val = theta2_data[val_indices]

# First held-out time marks the beginning of the future validation window.
train_time_end = float(t_train[-1, 0])
split_time = float(t_val[0, 0])

assert len(train_indices) + len(val_indices) == n_data
assert len(np.intersect1d(train_indices, val_indices)) == 0
assert train_time_end <= split_time

# ------------------------------------------------------------
# Characteristic angle scale: TRAINING ANGLES ONLY.
# ------------------------------------------------------------
theta_scale = max(
    np.max(np.abs(theta1_train)),
    np.max(np.abs(theta2_train)),
)
theta_scale = max(theta_scale, 1e-3)

# PyTorch tensors used by the network.
t_train_torch = torch.tensor(t_train, dtype=torch.float32, device=device)
t_val_torch = torch.tensor(t_val, dtype=torch.float32, device=device)
theta1_train_torch = torch.tensor(theta1_train, dtype=torch.float32, device=device)
theta2_train_torch = torch.tensor(theta2_train, dtype=torch.float32, device=device)
theta1_val_torch = torch.tensor(theta1_val, dtype=torch.float32, device=device)
theta2_val_torch = torch.tensor(theta2_val, dtype=torch.float32, device=device)

# The full time tensor is for evaluation and plotting ONLY.
t_data_torch = torch.tensor(t_data, dtype=torch.float32, device=device)

print("Angle scale (training only):", theta_scale)
print(f"Total observations:      {n_data}")
print(f"Training observations:   {len(train_indices)} ({len(train_indices)/n_data:.1%})")
print(f"Validation observations: {len(val_indices)} ({len(val_indices)/n_data:.1%})")
print(f"Training time window:    [{t_min:.3f}, {train_time_end:.3f}] s")
print(f"Validation time window:  [{split_time:.3f}, {t_max:.3f}] s")


### Visualizing the 80/20 split

The next plots show which measurements are used for training and which are held out for validation.

The vertical line marks the point where the validation interval begins.

The shaded region is the final 20% of the experiment.

When we examine the final model, pay special attention to this shaded region: it shows how well the model predicts measurements it did not use directly during fitting.

In [ ]:
# ============================================================
# 7B. VISUALIZE THE TRAINING / VALIDATION SPLIT
# ============================================================

# Left pendulum
plt.figure(figsize=(12, 4.5))

plt.plot(
    t_data[:, 0],
    theta1_data[:, 0],
    linewidth=1,
    alpha=0.25,
    label="All measured data"
)

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=12,
    alpha=0.50,
    label=f"Training ({len(train_indices)} points)"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label=f"Validation ({len(val_indices)} points)"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Left Pendulum: 80/20 Training–Validation Split")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Right pendulum
plt.figure(figsize=(12, 4.5))

plt.plot(
    t_data[:, 0],
    theta2_data[:, 0],
    linewidth=1,
    alpha=0.25,
    label="All measured data"
)

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=12,
    alpha=0.50,
    label=f"Training ({len(train_indices)} points)"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label=f"Validation ({len(val_indices)} points)"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Right Pendulum: 80/20 Training–Validation Split")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 🔵 A Neural-Network Approximation for $\theta_1(t)$ and $\theta_2(t)$

Up to this point, everything has been standard data preparation and differential-equation modeling.

We now introduce the main computational tool: a **Physics-Informed Neural Network**, or PINN.

You do not need a machine-learning course to follow the main idea.

A neural network is simply being used here as a flexible function. Instead of guessing a formula such as

$$
\theta(t)=A\cos(\omega t)+B\sin(\omega t),
$$

we let the computer build flexible approximations

$$
\widehat{\theta}_1(t)
\quad\text{and}\quad
\widehat{\theta}_2(t).
$$

The hat means "predicted."

### Rescaling time

The physical time values are first converted to a new variable $\tau$ between $-1$ and $1$:

$$
\tau
=
2\frac{t-t_{\min}}{t_{\max}-t_{\min}}-1.
$$

This does not change the physical problem. It is only a numerical rescaling.

### Why add sine and cosine terms?

Pendulums oscillate. Because sine and cosine functions naturally describe oscillations, we give the network extra inputs such as

$$
\sin(\pi\tau),\quad
\sin(2\pi\tau),\quad
\cos(\pi\tau),\quad
\cos(2\pi\tau),\ldots
$$

These are called **Fourier features**.

With `n_freq = 30`, the network receives

$$
\tau,
\sin(n\pi\tau),
\cos(n\pi\tau),
\qquad n=1,\ldots,30.
$$

The purpose is not to change the differential equation. The Fourier features simply make it easier for the neural network to represent oscillatory functions.

Finally, the network returns two numbers for each time $t$:

$$
\begin{bmatrix}
\widehat{\theta}_1(t)\\
\widehat{\theta}_2(t)
\end{bmatrix}.
$$

So you can think of the neural network as our computer-generated approximation to the two unknown solution functions.

In [ ]:
# ============================================================
# 8. FOURIER FEATURE PINN
# ============================================================


class FourierPINN(nn.Module):

    def __init__(
        self,
        t_min,
        t_max,
        theta_scale,
        n_freq=30,
        hidden=128
    ):

        super().__init__()

        self.register_buffer(
            "t_min",
            torch.tensor(
                float(t_min),
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "t_max",
            torch.tensor(
                float(t_max),
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "theta_scale",
            torch.tensor(
                float(theta_scale),
                dtype=torch.float32
            )
        )

        frequencies = torch.arange(
            1,
            n_freq + 1,
            dtype=torch.float32
        )

        self.register_buffer(
            "frequencies",
            frequencies
        )

        # tau + sin features + cos features
        input_dim = 1 + 2*n_freq

        self.net = nn.Sequential(

            nn.Linear(input_dim, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, hidden),
            nn.Tanh(),

            nn.Linear(hidden, 2)
        )


    def forward(self, t):

        # -----------------------------------
        # Normalize physical time to [-1, 1]
        # -----------------------------------

        tau = (
            2.0
            * (t - self.t_min)
            / (self.t_max - self.t_min)
            - 1.0
        )


        # -----------------------------------
        # Fourier features
        # -----------------------------------

        phase = (
            np.pi
            * tau
            * self.frequencies.reshape(1, -1)
        )


        features = torch.cat(
            (
                tau,
                torch.sin(phase),
                torch.cos(phase)
            ),
            dim=1
        )


        # Network predicts dimensionless angle
        raw_output = self.net(features)

        # Convert back to physical radians
        output = self.theta_scale * raw_output

        return output


model = FourierPINN(
    t_min=t_min,
    t_max=t_max,
    theta_scale=theta_scale,
    n_freq=30,
    hidden=128
).to(device)


print()
print(model)

## 8. Unknown physical parameters

The differential equations contain three quantities that we want to estimate:

$$
K=\frac{k}{m},\qquad d_1,\qquad d_2.
$$

Their meanings are:

- $K$: strength of the spring coupling per unit mass;
- $d_1$: damping coefficient for the left pendulum;
- $d_2$: damping coefficient for the right pendulum.

The computer will adjust these values while it adjusts the predicted functions $\widehat{\theta}_1$ and $\widehat{\theta}_2$.

### Why force the parameters to be positive?

For this model, negative spring strength or negative damping would not have the physical meaning we want. Therefore, the code represents each parameter using a positive transformation called the **softplus** function:

$$
\operatorname{softplus}(x)=\log(1+e^x).
$$

Since

$$
\operatorname{softplus}(x)>0,
$$

the estimates of $K$, $d_1$, and $d_2$ remain positive during training.

The starting guesses are

$$
K=1.0,\qquad d_1=0.02,\qquad d_2=0.02.
$$

These are only starting values. The computer will update them later.

For this lesson, the key point is:

> The physical constants are treated as **unknown parameters in the differential equations**, and we estimate them from the observed motion.

In [ ]:
# ============================================================
# 9. TRAINABLE PHYSICAL PARAMETERS
# ============================================================
#
# Instead of assuming k = 1, estimate k/m.
#
# Let
#
#     K = k/m
#
# Also estimate linear damping:
#
#     d1 theta1'
#     d2 theta2'
#
# softplus guarantees:
#
#     K > 0
#     d1 > 0
#     d2 > 0
#
# ============================================================

def inverse_softplus(x):

    x = torch.tensor(
        x,
        dtype=torch.float32,
        device=device
    )

    return torch.log(
        torch.exp(x) - 1.0
    )


# Reasonable starting guesses

raw_K = nn.Parameter(
    inverse_softplus(1.0)
)

raw_d1 = nn.Parameter(
    inverse_softplus(0.02)
)

raw_d2 = nn.Parameter(
    inverse_softplus(0.02)
)


def physical_parameters():

    K = F.softplus(raw_K)

    d1 = F.softplus(raw_d1)
    d2 = F.softplus(raw_d2)

    return K, d1, d2


## 9. Check whether the prediction satisfies the ODEs

This is where the differential equations become part of the learning process.

The neural network gives predicted functions

$$
\widehat{\theta}_1(t),
\qquad
\widehat{\theta}_2(t).
$$

PyTorch can differentiate these functions automatically, so it can compute

$$
\widehat{\theta}_1',
\quad
\widehat{\theta}_1'',
\quad
\widehat{\theta}_2',
\quad
\widehat{\theta}_2''.
$$

Now substitute the predictions into the two differential equations.

For the left pendulum, define

$$
R_1(t)
=
\widehat{\theta}_1''
+d_1\widehat{\theta}_1'
+\frac{g}{l_1}\widehat{\theta}_1
+\frac{K}{l_1}
(l_1\widehat{\theta}_1-l_2\widehat{\theta}_2).
$$

For the right pendulum,

$$
R_2(t)
=
\widehat{\theta}_2''
+d_2\widehat{\theta}_2'
+\frac{g}{l_2}\widehat{\theta}_2
-\frac{K}{l_2}
(l_1\widehat{\theta}_1-l_2\widehat{\theta}_2).
$$

The quantities $R_1$ and $R_2$ are called **residuals**.

If the predicted functions satisfied the ODEs exactly, then

$$
R_1(t)=0,
\qquad
R_2(t)=0
$$

for every time $t$.

So a small residual means that the predicted motion is close to obeying the differential-equation model.

In [ ]:
# ============================================================
# 10. AUTOMATIC DIFFERENTIATION + PHYSICS
# ============================================================

def physics_residual(model, t):

    # Need derivatives with respect to physical t
    t = t.clone().detach().requires_grad_(True)

    output = model(t)

    theta1 = output[:, 0:1]
    theta2 = output[:, 1:2]


    # ========================================================
    # First derivatives
    # ========================================================

    theta1_t = torch.autograd.grad(
        theta1,
        t,
        grad_outputs=torch.ones_like(theta1),
        create_graph=True
    )[0]


    theta2_t = torch.autograd.grad(
        theta2,
        t,
        grad_outputs=torch.ones_like(theta2),
        create_graph=True
    )[0]


    # ========================================================
    # Second derivatives
    # ========================================================

    theta1_tt = torch.autograd.grad(
        theta1_t,
        t,
        grad_outputs=torch.ones_like(theta1_t),
        create_graph=True
    )[0]


    theta2_tt = torch.autograd.grad(
        theta2_t,
        t,
        grad_outputs=torch.ones_like(theta2_t),
        create_graph=True
    )[0]


    # ========================================================
    # Physical parameters
    # ========================================================

    K, d1, d2 = physical_parameters()


    # ========================================================
    # Spring extension
    #
    # l1 theta1 - l2 theta2
    # ========================================================

    coupling = (
        l1 * theta1
        -
        l2 * theta2
    )


    # ========================================================
    # ODE #1
    #
    # theta1''
    # + d1 theta1'
    # + g/l1 theta1
    # + K/l1 (l1 theta1-l2 theta2)
    # = 0
    # ========================================================

    residual1 = (
        theta1_tt
        +
        d1 * theta1_t
        +
        (g/l1) * theta1
        +
        (K/l1) * coupling
    )


    # ========================================================
    # ODE #2
    # ========================================================

    residual2 = (
        theta2_tt
        +
        d2 * theta2_t
        +
        (g/l2) * theta2
        -
        (K/l2) * coupling
    )


    return (
        residual1,
        residual2,
        theta1,
        theta2,
        theta1_t,
        theta2_t
    )


## 10. Measure how well the prediction matches the data

We need a numerical way to measure the difference between the model prediction and the experimental measurements.

At a training time $t_j$, compare

$$
\widehat{\theta}_1(t_j)
\quad\text{with}\quad
\theta_{1,j},
$$

and similarly for the second pendulum.

The code uses the **mean squared error**: square the differences and average them.

In simplified form,

$$
\mathcal{L}_{\text{data}}
=
\text{average squared error for pendulum 1}
+
\text{average squared error for pendulum 2}.
$$

The errors are divided by `theta_scale` so that the loss is dimensionless.

A small data loss means

$$
\boxed{\text{prediction is close to the measured angles}}.
$$

The training loss uses only the first 80% of the observations.

We also calculate the same type of error on the final 20%, called the **validation loss**, but we do not use that value to update the model.

In [ ]:
# ============================================================
# 11. TRAINING + VALIDATION DATA LOSS
# ============================================================

def normalized_data_mse(
    t_tensor,
    theta1_tensor,
    theta2_tensor
):
    """Dimensionless MSE for any supplied data subset."""

    prediction = model(t_tensor)

    pred1 = prediction[:, 0:1]
    pred2 = prediction[:, 1:2]

    error1 = (
        pred1 - theta1_tensor
    ) / theta_scale

    error2 = (
        pred2 - theta2_tensor
    ) / theta_scale

    return (
        torch.mean(error1**2)
        +
        torch.mean(error2**2)
    )


def data_loss():
    """Training loss: used for backpropagation."""

    return normalized_data_mse(
        t_train_torch,
        theta1_train_torch,
        theta2_train_torch
    )


def validation_loss():
    """Held-out loss: evaluation only, never backpropagated."""

    with torch.no_grad():
        return normalized_data_mse(
            t_val_torch,
            theta1_val_torch,
            theta2_val_torch
        )


## 11. Measure how well the prediction satisfies the physics

Matching the data is only one goal. We also want the prediction to satisfy the differential equations.

At many randomly chosen times, we calculate the ODE residuals

$$
R_1(t),\qquad R_2(t).
$$

If the ODEs are satisfied exactly, both residuals are zero.

The code therefore defines a **physics loss** by averaging the squared residuals:

$$
\mathcal{L}_{\text{physics}}
\approx
\text{average of }R_1(t)^2
+
\text{average of }R_2(t)^2.
$$

The actual code first divides the residuals by characteristic acceleration scales,

$$
\frac{g}{l_1}\theta_{\text{scale}}
\qquad\text{and}\qquad
\frac{g}{l_2}\theta_{\text{scale}},
$$

so that the numbers being compared are dimensionless.

A small physics loss means

$$
\boxed{\text{the predicted functions nearly satisfy the ODE system}}.
$$

The random times used here are called **collocation points**. They do not have to be measurement times. They are simply points where we check the differential equations.

In [ ]:
# ============================================================
# 12. PHYSICS LOSS
# ============================================================

def physics_loss(
    n_physics=512
):

    # Generate new random collocation points every iteration.
    #
    # This is usually better and cheaper than using thousands
    # of permanently fixed points.

    # Known ODEs can be enforced at future times without using
    # the held-out future MEASUREMENTS. This is physics-assisted
    # forecasting, not training on validation labels.
    # For the stricter training-time-only-physics experiment,
    # replace t_max below with train_time_end.
    physics_time_end = t_max
    t_phys = (
        t_min
        +
        (physics_time_end - t_min)
        * torch.rand(
            n_physics,
            1,
            device=device
        )
    )


    r1, r2, _, _, _, _ = physics_residual(
        model,
        t_phys
    )


    # Characteristic acceleration:
    #
    #       (g/L) theta_scale
    #
    # Divide residual by this so the physics loss is
    # dimensionless.

    acceleration_scale_1 = (
        (g/l1) * theta_scale
    )

    acceleration_scale_2 = (
        (g/l2) * theta_scale
    )


    r1_normalized = (
        r1 / acceleration_scale_1
    )

    r2_normalized = (
        r2 / acceleration_scale_2
    )


    loss = (
        torch.mean(r1_normalized**2)
        +
        torch.mean(r2_normalized**2)
    )

    return loss

# 🔵 Stage 1: Learning from the Data

Training is done in two stages.

In **Stage 1**, we temporarily ignore the differential equations and ask the neural network only to fit the training measurements.

Symbolically,

$$
\text{minimize }\mathcal{L}_{\text{data}}.
$$

During this stage:

- the neural-network approximation changes;
- only the first 80% of the measured angles are used for fitting;
- $K$, $d_1$, and $d_2$ are not changed yet.

### Why start this way?

It gives the network a reasonable first approximation to the observed oscillations before we ask it to satisfy the more restrictive physical model.

Think of this as making an initial guess for the solution before adding the ODE information.

In [ ]:
# ============================================================
# 13. STAGE 1:
#     DATA-ONLY PRETRAINING ON TRAINING SET
# ============================================================

warmup_epochs = 10000
validation_every_stage1 = 100

optimizer_data = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

warmup_history = []
warmup_val_epochs = []
warmup_val_history = []

print()
print("========================================")
print("STAGE 1: DATA-ONLY TRAINING")
print("========================================")

for epoch in range(warmup_epochs):

    optimizer_data.zero_grad()

    # Uses only the training observations.
    loss_d = data_loss()

    loss_d.backward()
    optimizer_data.step()

    warmup_history.append(loss_d.item())

    # Validation is evaluation only; no gradient is taken.
    if (
        epoch == 0
        or (epoch + 1) % validation_every_stage1 == 0
        or epoch + 1 == warmup_epochs
    ):
        loss_v = validation_loss()
        warmup_val_epochs.append(epoch + 1)
        warmup_val_history.append(loss_v.item())

    if (epoch + 1) % 1000 == 0:
        loss_v_print = validation_loss().item()

        print(
            f"Epoch {(epoch+1):5d} | "
            f"Training loss = {loss_d.item():.4e} | "
            f"Validation loss = {loss_v_print:.4e}"
        )


### Check the Stage 1 fit

The next plots compare the data with the neural-network prediction after data-only training.

Look separately at:

- the training points, which the network used;
- the validation points, which the network did not use;
- the continuous predicted curve.

Questions to consider:

1. Does the curve follow the training measurements?
2. Does it continue to oscillate in a reasonable way?
3. How well does it predict the later validation measurements?

At this stage, the network has learned from data, but we have not yet required it to satisfy the coupled-pendulum ODEs.

In [ ]:

# ============================================================
# 14. CHECK DATA-ONLY FIT
#     PLOT LEFT AND RIGHT PENDULUMS SEPARATELY
# ============================================================

model.eval()

with torch.no_grad():
    prediction_warmup = (
        model(t_data_torch)
        .cpu()
        .numpy()
    )

model.train()

# ------------------------------------------------------------
# LEFT PENDULUM
# ------------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=10,
    alpha=0.35,
    label="Left training data"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=10,
    label="Left validation data"
)

plt.plot(
    t,
    prediction_warmup[:, 0],
    color="red",
    linewidth=2,
    label="Left NN after warm-up"
)

plt.axvspan(
    split_time, t_max,
    color="gold",
    alpha=0.12,
    label="Future validation window"
)

plt.axvline(
    split_time,
    color="black",
    linestyle=":",
    linewidth=1.4
)

plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Stage 1: Left Pendulum — Training vs Validation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# RIGHT PENDULUM
# ------------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=10,
    alpha=0.35,
    label="Right training data"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=10,
    label="Right validation data"
)

plt.plot(
    t,
    prediction_warmup[:, 1],
    color="red",
    linewidth=2,
    label="Right NN after warm-up"
)

plt.axvspan(
    split_time, t_max,
    color="gold",
    alpha=0.12,
    label="Future validation window"
)

plt.axvline(
    split_time,
    color="black",
    linestyle=":",
    linewidth=1.4
)

plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Stage 1: Right Pendulum — Training vs Validation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## ❓ Question 2: How Many Fourier Features Do We Need?

The neural network uses `n_freq` to decide how many sine and cosine features are included.

The current notebook uses

```python
n_freq=30
```

At first, it may seem natural to think that using more Fourier features should always improve the model. However, that is not necessarily true.

### Your Task

Find the code cell that defines `n_freq`.

Try the following values one at a time in the `FourierPINN(...)` call:

```python
n_freq=0
n_freq=5
n_freq=30
```

For each choice, re-run the neural-network definition and **Stage 1** training.

Record the **final training loss** and **final validation loss** for each value of `n_freq`. You may organize your results in a table such as

| `n_freq` | Final training loss | Final validation loss |
|---:|---:|---:|
| 0 |  |  |
| 5 |  |  |
| 30 |  |  |

### Questions to Answer

1. Which value of `n_freq` gives the smallest training loss?
2. Which value gives the smallest validation loss?
3. Does increasing `n_freq` always improve the validation loss?
4. In this experiment, how does `n_freq=5` compare with `n_freq=30`?
5. Why is the validation loss especially useful when deciding how many Fourier features to use?

### What to Notice

Do not assume that a larger or smaller value of `n_freq` will automatically be better.

Use the **training loss** and especially the **validation loss** to compare the models. A small training loss tells us how well the network fits the data it has seen, while the validation loss tells us how well it predicts the held-out portion of the experiment.

For this data set, the important lesson is that model selection should be based on the numerical evidence rather than on the number of Fourier features alone.

**Your observations:**  
*Write here.*

---


### Before You Continue

Based on your experiment above, choose the value of `n_freq` that gave the better validation performance.

Before moving on, **go back to the neural-network definition cell**, set `n_freq` to that chosen value, and then re-run the neural-network definition and **Stage 1** training.

Use this better value of `n_freq` for the next experiment.


# 🔵 Stage 2: Adding the Differential Equations

Now we want both goals at the same time:

1. match the experimental data;
2. satisfy the ODE system.

We combine the two errors into one total loss:

$$
\mathcal{L}_{\text{total}}
=
\mathcal{L}_{\text{data}}
+
\lambda_p\mathcal{L}_{\text{physics}}.
$$

The number $\lambda_p$ tells us how much weight to give the differential equations.

In this notebook,

$$
\lambda_p=2.
$$

So training now searches for functions that balance

$$
\boxed{\text{agreement with data}}
\qquad\text{and}\qquad
\boxed{\text{agreement with the ODEs}}.
$$

This is the central idea of a **Physics-Informed Neural Network**.

### Stage 2: Fit the data and the physics together

In Stage 2, the computer updates

- the neural-network approximation,
- the coupling parameter $K$,
- the damping parameters $d_1$ and $d_2$.

The quantity being minimized is

$$
\mathcal{L}_{\text{total}}
=
\mathcal{L}_{\text{data}}
+
\lambda_p\mathcal{L}_{\text{physics}}.
$$

The measured validation angles are still held out.

This stage is especially interesting from a differential-equations point of view because the data are now helping us estimate unknown coefficients in the ODE system.

In [ ]:
# ============================================================
# 15. STAGE 2:
#     JOINT TRAINING DATA + PHYSICS
# ============================================================

physics_epochs = 10000
validation_every_stage2 = 100


optimizer_pinn = torch.optim.Adam(
    list(model.parameters())
    +
    [raw_K, raw_d1, raw_d2],
    lr=5e-4
)

total_history = []
data_history = []
physics_history = []
validation_epochs = []
validation_history = []
K_history = []
d1_history = []
d2_history = []

print()
print("========================================")
print("STAGE 2: PINN TRAINING")
print("========================================")

for epoch in range(physics_epochs):

    optimizer_pinn.zero_grad()

    # ----------------------------------------
    # Training observations only
    # ----------------------------------------
    loss_d = data_loss()

    # ----------------------------------------
    # Physics over the full horizon (NO future measured angles)
    # ----------------------------------------
    loss_p = physics_loss(
        n_physics=512
    )

    lambda_p = 0.5

    total_loss = (
        loss_d
        +
        lambda_p * loss_p
    )

    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(model.parameters())
        +
        [raw_K, raw_d1, raw_d2],
        max_norm=10.0
    )

    optimizer_pinn.step()

    # ----------------------------------------
    # Save training histories
    # ----------------------------------------
    K_now, d1_now, d2_now = physical_parameters()

    total_history.append(total_loss.item())
    data_history.append(loss_d.item())
    physics_history.append(loss_p.item())

    K_history.append(K_now.item())
    d1_history.append(d1_now.item())
    d2_history.append(d2_now.item())

    # ----------------------------------------
    # Held-out validation history
    # ----------------------------------------
    if (
        epoch == 0
        or (epoch + 1) % validation_every_stage2 == 0
        or epoch + 1 == physics_epochs
    ):
        loss_v = validation_loss()
        validation_epochs.append(epoch + 1)
        validation_history.append(loss_v.item())


  # ----------------------------------------
    # Print progress every 1000 epochs
    # ----------------------------------------
    if (epoch + 1) % 1000 == 0:

        loss_v_print = validation_loss().item()

        print(
            f"Epoch {(epoch + 1):5d} | "
            f"Training data loss = {loss_d.item():.3e} | "
            f"Physics loss = {loss_p.item():.3e} | "
            f"Total training loss = {total_loss.item():.3e} | "
            f"Validation loss = {loss_v_print:.3e}"
        )

        print(
            f"        Coupling parameter K = k/m = {K_now.item():.5f} | "
            f"Damping d1 = {d1_now.item():.5f} | "
            f"Damping d2 = {d2_now.item():.5f}"
        )

## 14. Compare the errors during training

The next plots show how the errors change as the model is trained.

An **epoch** is one update step in the optimization process.

We will compare:

- training data loss,
- validation data loss,
- physics loss,
- total loss.

What should we hope to see?

- A decreasing training loss means the model is fitting the known data better.
- A decreasing physics loss means the prediction is satisfying the ODEs more closely.
- A small validation loss means the model is also predicting the held-out measurements well.

The validation error does not have to decrease every time the training error decreases. That is one reason we keep validation data separate.

In [ ]:
# ============================================================
# 16. TRAINING + VALIDATION LOSS COMPARISON
# ============================================================

# ------------------------------------------------------------
# Stage 1: training vs validation
# ------------------------------------------------------------
plt.figure(figsize=(11, 5))

stage1_epochs = np.arange(1, len(warmup_history) + 1)

plt.semilogy(
    stage1_epochs,
    warmup_history,
    linewidth=1.8,
    label="Training data loss"
)

plt.semilogy(
    warmup_val_epochs,
    warmup_val_history,
    'o-',
    markersize=4,
    linewidth=1.5,
    label="Validation data loss"
)

best_stage1_i = int(np.argmin(warmup_val_history))
best_stage1_epoch = warmup_val_epochs[best_stage1_i]
best_stage1_val = warmup_val_history[best_stage1_i]

plt.scatter(
    [best_stage1_epoch],
    [best_stage1_val],
    s=90,
    marker="*",
    zorder=5,
    label=f"Best validation epoch = {best_stage1_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized data loss")
plt.title("Stage 1: Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Stage 2: compare ONLY training data loss and validation loss
# This makes the generalization gap easy to see.
# ------------------------------------------------------------
plt.figure(figsize=(11, 5))

stage2_epochs = np.arange(1, len(data_history) + 1)

plt.semilogy(
    stage2_epochs,
    data_history,
    linewidth=1.6,
    label="Training data loss"
)

plt.semilogy(
    validation_epochs,
    validation_history,
    'o-',
    markersize=4,
    linewidth=1.5,
    label="Validation data loss"
)

best_stage2_i = int(np.argmin(validation_history))
best_stage2_epoch = validation_epochs[best_stage2_i]
best_stage2_val = validation_history[best_stage2_i]

plt.scatter(
    [best_stage2_epoch],
    [best_stage2_val],
    s=90,
    marker="*",
    zorder=5,
    label=f"Best validation epoch = {best_stage2_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized data loss")
plt.title("Stage 2: Training vs Validation Data Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Stage 2: total and physics losses
# Kept separate so they do not visually hide the validation curve.
# ------------------------------------------------------------
plt.figure(figsize=(11, 5))

plt.semilogy(
    stage2_epochs,
    total_history,
    linewidth=1.6,
    label="Total PINN loss"
)

plt.semilogy(
    stage2_epochs,
    physics_history,
    linewidth=1.6,
    label="Physics loss"
)
plt.semilogy(
    stage2_epochs,
    data_history,
    linewidth=1.6,
    label="Data loss"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized loss")
plt.title("Stage 2: Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---

## ❓ Question 3: How Much Should We Trust the Physics?

In Stage 2, the total loss is

$$
\mathcal{L}_{\text{total}}
=
\mathcal{L}_{\text{data}}
+
\lambda_p\mathcal{L}_{\text{physics}}.
$$

The parameter $\lambda_p$ controls how strongly the differential equations influence the training.

### Your Task

Keep the better value of `n_freq` that you selected in Question 2.

Compare

```python
lambda_p = 0.0
lambda_p = 0.5
lambda_p = 2.0
```

For each value of $\lambda_p$:

1. **Go back to the neural-network definition cell** and make sure it still uses the `n_freq` value you selected in Question 2.
2. Re-run the neural-network definition.
3. Re-run **Stage 1** so that each experiment begins from the same type of data-only fit.
4. Go to the Stage 2 cell and set the desired value of `lambda_p`.
5. Run Stage 2.
6. Record the final training data loss, validation loss, physics loss, and estimated $K=k/m$.

You may use a table like this:

| $\lambda_p$ | Training data loss | Validation loss | Physics loss | Final $K=k/m$ |
|---:|---:|---:|---:|---:|
| 0.0 |  |  |  |  |
| 0.5 |  |  |  |  |
| 2.0 |  |  |  |  |

### Questions to Answer

1. What happens when $\lambda_p=0$?
2. As $\lambda_p$ increases, what happens to the physics loss?
3. Which value gives the smallest validation loss?
4. Does the smallest physics loss necessarily give the smallest validation loss?
5. Why do we need both the data loss and the physics loss?

**Your observations:**  
*Write here.*

---


### Before You Continue

Based on your experiment above, choose the value of $\lambda_p$ that gives the better overall result.

Use the validation loss as your main guide, while also checking that the physics loss remains reasonably small.

Before moving on, **go back and re-run Stage 1**, then set Stage 2 to your chosen value of $\lambda_p$ and run Stage 2 again.

Use this better value of $\lambda_p$ for the next experiment.


## 15. Watch the estimated physical parameters

While Stage 2 runs, the model repeatedly updates

$$
K=\frac{k}{m},\qquad d_1,\qquad d_2.
$$

The next plots show these estimates as functions of the training epoch.

If a parameter curve begins to level off, that suggests the estimate is stabilizing.

If it continues to change greatly near the end, the estimation may not yet have converged.

This is an example of an **inverse problem**:

> Instead of knowing the coefficients of the differential equation and solving for the motion, we observe the motion and try to determine some of the coefficients.

In [ ]:
# ============================================================
# 17. PHYSICAL-PARAMETER HISTORY
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    K_history,
    label=r"$K=k/m$"
)

plt.xlabel("Epoch")
plt.ylabel(r"$k/m$ [s$^{-2}$]")
plt.title("Estimated Coupling Parameter")
plt.grid(True)
plt.legend()

plt.show()


plt.figure(figsize=(10, 5))

plt.plot(
    d1_history,
    label=r"$d_1$"
)

plt.plot(
    d2_history,
    label=r"$d_2$"
)

plt.xlabel("Epoch")
plt.ylabel(r"Damping [s$^{-1}$]")
plt.title("Estimated Damping")
plt.grid(True)
plt.legend()

plt.show()

## 16. Final prediction and validation

After training, we evaluate the model at 3,000 closely spaced time values. This gives smooth curves for

$$
\widehat{\theta}_1(t)
\qquad\text{and}\qquad
\widehat{\theta}_2(t).
$$

We also compare the predictions with the training and validation measurements.

Several numerical errors are reported. Two useful ones are:

### Root Mean Square Error (RMSE)

RMSE measures a typical size of the prediction error. Smaller is better.

### Relative $L^2$ error

This compares the overall size of the prediction error with the overall size of the measured signal.

The most important idea is not the name of each metric. Ask:

> How close is the predicted motion to the measurements that were not used directly for fitting?

Remember that the final 20% of measured angles were held out from the data-fitting loss.

In [ ]:
# ============================================================
# 18. FINAL PINN PREDICTIONS + VALIDATION METRICS
# ============================================================

t_test = np.linspace(
    t_min,
    t_max,
    3000,
    dtype=np.float32
).reshape(-1, 1)


t_test_torch = torch.tensor(
    t_test,
    dtype=torch.float32,
    device=device
)

model.eval()

with torch.no_grad():

    prediction = (
        model(t_test_torch)
        .cpu()
        .numpy()
    )

    pred_train = (
        model(t_train_torch)
        .cpu()
        .numpy()
    )

    pred_val = (
        model(t_val_torch)
        .cpu()
        .numpy()
    )


theta1_pinn = prediction[:, 0]
theta2_pinn = prediction[:, 1]

# ------------------------------------------------------------
# Final training and validation metrics
# ------------------------------------------------------------

train_loss_final = np.mean(
    ((pred_train[:, 0] - theta1_train[:, 0]) / theta_scale)**2
) + np.mean(
    ((pred_train[:, 1] - theta2_train[:, 0]) / theta_scale)**2
)

val_loss_final = np.mean(
    ((pred_val[:, 0] - theta1_val[:, 0]) / theta_scale)**2
) + np.mean(
    ((pred_val[:, 1] - theta2_val[:, 0]) / theta_scale)**2
)

rmse_train_left = np.sqrt(
    np.mean((pred_train[:, 0] - theta1_train[:, 0])**2)
)
rmse_train_right = np.sqrt(
    np.mean((pred_train[:, 1] - theta2_train[:, 0])**2)
)

rmse_val_left = np.sqrt(
    np.mean((pred_val[:, 0] - theta1_val[:, 0])**2)
)
rmse_val_right = np.sqrt(
    np.mean((pred_val[:, 1] - theta2_val[:, 0])**2)
)

relative_l2_train_left = (
    np.linalg.norm(pred_train[:, 0] - theta1_train[:, 0])
    /
    max(np.linalg.norm(theta1_train[:, 0]), 1e-12)
)

relative_l2_train_right = (
    np.linalg.norm(pred_train[:, 1] - theta2_train[:, 0])
    /
    max(np.linalg.norm(theta2_train[:, 0]), 1e-12)
)

relative_l2_val_left = (
    np.linalg.norm(pred_val[:, 0] - theta1_val[:, 0])
    /
    max(np.linalg.norm(theta1_val[:, 0]), 1e-12)
)

relative_l2_val_right = (
    np.linalg.norm(pred_val[:, 1] - theta2_val[:, 0])
    /
    max(np.linalg.norm(theta2_val[:, 0]), 1e-12)
)

print()
print("========================================")
print("FINAL TRAINING / VALIDATION PERFORMANCE")
print("========================================")
print(f"Normalized training loss   = {train_loss_final:.6e}")
print(f"Normalized validation loss = {val_loss_final:.6e}")
print()
print(f"Training RMSE (left pendulum)     = {rmse_train_left:.6e} rad")
print(f"Validation RMSE (left pendulum)   = {rmse_val_left:.6e} rad")
print(f"Training RMSE (right pendulum)    = {rmse_train_right:.6e} rad")
print(f"Validation RMSE (right pendulum)  = {rmse_val_right:.6e} rad")
print()
print(f"Training relative L2 error (left)    = {relative_l2_train_left:.6e}")
print(f"Validation relative L2 error (left)  = {relative_l2_val_left:.6e}")
print(f"Training relative L2 error (right)   = {relative_l2_train_right:.6e}")
print(f"Validation relative L2 error (right) = {relative_l2_val_right:.6e}")


### Left pendulum

The next plot compares the final prediction

$$
\widehat{\theta}_1(t)
$$

with the measured angle of the left pendulum.

Pay particular attention to the shaded validation region.

A good prediction should continue the oscillatory motion and remain reasonably close to the held-out measurements.

In [ ]:
# ============================================================
# 19. LEFT PENDULUM: TRAINING + VALIDATION + RESIDUALS
# ============================================================

# Main trajectory comparison
plt.figure(figsize=(12, 5))

plt.scatter(
    t_train[:, 0],
    theta1_train[:, 0],
    s=12,
    alpha=0.25,
    label="Training measurements"
)

plt.scatter(
    t_val[:, 0],
    theta1_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label="Validation measurements"
)

plt.plot(
    t_test[:, 0],
    theta1_pinn,
    color = "red",
    linewidth=2,
    label="PINN trajectory"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_1$ [rad]")
plt.title("Left Pendulum: Held-Out Validation Points vs PINN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### Right pendulum

Now we make the same comparison for

$$
\widehat{\theta}_2(t).
$$

Again, the measurements in the shaded validation interval were not included in the data-fitting loss.

Comparing both pendulums is important because the two differential equations are coupled: an inaccurate model for one pendulum can affect the prediction for the other.

In [ ]:
# ============================================================
# 20. RIGHT PENDULUM: TRAINING + VALIDATION + RESIDUALS
# ============================================================

# Main trajectory comparison
plt.figure(figsize=(12, 5))

plt.scatter(
    t_train[:, 0],
    theta2_train[:, 0],
    s=12,
    alpha=0.25,
    label="Training measurements"
)

plt.scatter(
    t_val[:, 0],
    theta2_val[:, 0],
    s=12,
    # marker="x",
    linewidths=1.4,
    label="Validation measurements"
)


plt.plot(
    t_test[:, 0],
    theta2_pinn,
    color = "red",
    linewidth=2,
    label="PINN trajectory"
)

plt.axvspan(split_time, t_max, color="gold", alpha=0.12,
            label="Future validation window")
plt.axvline(split_time, color="black", linestyle=":", linewidth=1.4)
plt.xlabel("Time [s]")
plt.ylabel(r"$\theta_2$ [rad]")
plt.title("Right Pendulum: Held-Out Validation Points vs PINN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 💡 Final Results and Main Takeaway

Finally, we print the estimated values of

$$
K=\frac{k}{m},
\qquad
d_1,
\qquad
d_2.
$$

Their units are

$$
K:\ \mathrm{s}^{-2},
\qquad
d_1,d_2:\ \mathrm{s}^{-1}.
$$

These numbers summarize the physical behavior inferred from the experiment:

- $K$ describes the strength of the spring coupling;
- $d_1$ describes damping of the left pendulum;
- $d_2$ describes damping of the right pendulum.

---

## Main takeaway

In a traditional differential equations problem, the coefficients are usually given and we solve for the unknown function.

Here, some coefficients are unknown and the measurements are imperfect. The PINN uses two sources of information:

$$
\boxed{\text{experimental data}}
\qquad+\qquad
\boxed{\text{differential equations}}.
$$

The goal is to find functions $\widehat{\theta}_1(t)$ and $\widehat{\theta}_2(t)$ that fit the observations while also behaving like solutions of the coupled ODE system.

This gives us a modern example of how differential equations can be used not only to **predict motion**, but also to **learn unknown physical parameters from data**.

In [ ]:
# ============================================================
# 22. FINAL PARAMETER ESTIMATES
# ============================================================

K_final, d1_final, d2_final = (
    physical_parameters()
)


print()
print("========================================")
print("FINAL ESTIMATED PARAMETERS")
print("========================================")

print(
    "Coupling parameter K = k/m =",
    K_final.item(),
    "s^-2"
)

print(
    "Damping coefficient d1 =",
    d1_final.item(),
    "s^-1"
)

print(
    "Damping coefficient d2 =",
    d2_final.item(),
    "s^-1"
)

---

## ❓ Question 4: Do the Initial Parameter Guesses Matter?

Now investigate whether the final parameter estimates depend strongly on the starting guess.

Keep the better values of `n_freq` and $\lambda_p$ that you selected in Questions 2 and 3.

### Your Task

Compare the following initial guesses for $K=k/m$:

$$
K_0=0.2,\qquad 1.0,\qquad 2.0.
$$

Keep the damping guesses fixed at

$$
d_{1,0}=d_{2,0}=0.02.
$$

For each initial value of $K$:

1. **Go back to the neural-network definition cell** and make sure it uses your selected `n_freq`.
2. Re-run the neural-network definition and **Stage 1**.
3. Go back to the parameter-definition cell and change only the starting value of $K$:
   ```python
   raw_K = nn.Parameter(inverse_softplus(0.2))
   ```
   or
   ```python
   raw_K = nn.Parameter(inverse_softplus(1.0))
   ```
   or
   ```python
   raw_K = nn.Parameter(inverse_softplus(2.0))
   ```
4. Keep the damping starting values unchanged.
5. In Stage 2, keep the better value of $\lambda_p$ that you selected in Question 3.
6. Run Stage 2 and record the final results.

Record your results:

| Initial $K_0$ | Final $K$ | Final $d_1$ | Final $d_2$ | Validation loss |
|---:|---:|---:|---:|---:|
| 0.2 |  |  |  |  |
| 1.0 |  |  |  |  |
| 2.0 |  |  |  |  |

### Questions to Answer

1. Do the different starting guesses lead to similar final values of $K$?
2. How do the final values of $d_1$ and $d_2$ change?
3. How does the validation loss compare in each case?
4. What does this experiment tell you about estimating unknown parameters from data?

**Your observations:**  
*Write here.*

---


---

## ❓ Final Comparison: How Do the Two Methods Compare?

In the previous lesson, you estimated the coupling parameter

$$
K=\frac{k}{m}
$$

using the observed natural frequencies of the coupled pendulum system.

In this lesson, you estimated the same parameter using a Physics-Informed Neural Network (PINN), together with the damping parameters $d_1$ and $d_2$.

### Your Task

Record the value of $K=k/m$ from the previous Fourier/natural-frequency lesson and the final value from this PINN lesson. Use the parameter values produced by your own runs.

| Method | Estimated $K=k/m$ |
|---|---:|
| Fourier / natural-frequency method |  |
| PINN method |  |

### Questions to Answer

1. Are the two estimates of $K=k/m$ close to each other?
2. Which assumptions are different between the two methods?
3. The previous method did not explicitly estimate damping, while the PINN estimates $d_1$ and $d_2$. How might this affect the estimate of $K$?
4. If the two values are not exactly the same, does that necessarily mean one method is wrong? Explain.
5. Which method seems more flexible for working with experimental data, and why?

**Your comparison:**  
*Write here.*
